In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [3]:
code = 'NEXG'
market = 'V'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=code,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [4]:
dsv_timeseries_df = request_api.get_stock_time_series_data(
    code=code,
    market=market,
    start=start,
    end=end
)
dsv_timeseries_df

取得件数: 4447


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1187175,NEXG,V,2008-08-19,18.000000,21.000000,18.000000,21.000000,650,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1187176,NEXG,V,2008-08-20,16.799999,18.000000,16.799999,18.000000,58,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1187177,NEXG,V,2008-08-21,16.799999,16.799999,16.799999,16.799999,8,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1187178,NEXG,V,2008-08-22,12.600000,16.200001,10.800000,15.600000,1900,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1187179,NEXG,V,2008-08-25,7.800000,13.800000,6.240000,13.800000,625,17.040,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4442,1191617,NEXG,V,2026-05-04,1.430000,1.570000,1.430000,1.570000,228500,1.506,...,1.681094,1.491706,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4443,1191618,NEXG,V,2026-05-05,1.420000,1.470000,1.400000,1.440000,255000,1.476,...,1.681437,1.489763,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4444,1191619,NEXG,V,2026-05-06,1.560000,1.570000,1.470000,1.490000,320600,1.486,...,1.678544,1.501456,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4445,1191620,NEXG,V,2026-05-07,1.480000,1.650000,1.470000,1.620000,359200,1.520,...,1.678544,1.501456,False,NaN,NaN,NaN,NaN,-5.833333,NaN,False


In [5]:
def stock_prices_and_material_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_mat1: pd.DataFrame | None = None,
        df_mat2: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # mat1価格を統合
    if df_mat1 is not None:
        df_mat1_tmp = df_mat1.copy() if df_mat1 is not None else pd.DataFrame()
        if "date" not in df_mat1_tmp.columns:
            df_mat1_tmp = df_mat1_tmp.reset_index()
        df_mat1_tmp["date"] = pd.to_datetime(df_mat1_tmp["date"])
        df_mat1_tmp = df_mat1_tmp.set_index("date")
        df_mat1_tmp = df_mat1_tmp.loc[start:end]

    # mat2価格を統合
    if df_mat2 is not None:
        df_mat2_tmp = df_mat2.copy() if df_mat2 is not None else pd.DataFrame()
        if "date" not in df_mat2_tmp.columns:
            df_mat2_tmp = df_mat2_tmp.reset_index()
        df_mat2_tmp["date"] = pd.to_datetime(df_mat2_tmp["date"])
        df_mat2_tmp = df_mat2_tmp.set_index("date")
        df_mat2_tmp = df_mat2_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_mat1 is not None:
        df["MA5_MAT1"] = df_mat1_tmp["ma5"].reindex(df.index)
        df["MA25_MAT1"] = df_mat1_tmp["ma25"].reindex(df.index)
    if df_mat2 is not None:
        df["MA5_MAT2"] = df_mat2_tmp["ma5"].reindex(df.index)
        df["MA25_MAT2"] = df_mat2_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT1（右軸） ---
    if df_mat1 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT1"],
                name="MAT1_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT1"],
                name="MAT1_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT2（左軸） ---
    if df_mat2 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT2"],
                name="MAT2_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT2"],
                name="MAT2_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [6]:
name = "Silver Mountain Resources Inc"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_material_prices(
    code=code,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_mat1=None,
    df_mat2=None
)
fig.show()

取得件数: 531


In [7]:
response = request_api.update_corp_finance_data(
    code=code,
    market=market
)
response

{'result': True}

In [8]:
nexg_financials_data = request_api.get_corp_financials_data(code=code, market=market)
nexg_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=code, market=market)
nexg_cash_flow_data = request_api.get_corp_cash_flow_data(code=code, market=market)
nexg_earnings_data = request_api.get_corp_earnings_data(code=code, market=market)
nexg_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=code, market=market)

In [9]:
# ４年分の財務データ
nexg_financials_data_df = pd.DataFrame(nexg_financials_data['results'])
# ４年分のバランスシート
nexg_balance_sheet_data_df = pd.DataFrame(nexg_balance_sheet_data['results'])
# ４年分のキャッシュフロー
nexg_cash_flow_data_df = pd.DataFrame(nexg_cash_flow_data['results'])
# ４年分の収益データ
nexg_earnings_data_df = pd.DataFrame(nexg_earnings_data['results'])
# ４年分の四半期収益データ
nexg_quarterly_earnings_data_df = pd.DataFrame(nexg_quarterly_earnings_data['results'])

In [10]:
"""
◆ 1. 株価・市場データ
• 現在株価（Price）
• 時価総額（Market Cap）
• 出来高（Volume）
• 52週高値・安値
• Beta（ボラティリティ指標）ß
"""
stock_prices_and_market_data = stock_prices_market_data.stock_prices_and_market_data(
    code=code,
    market=market,
    bs_df=nexg_balance_sheet_data_df
)
stock_prices_and_market_data.to_markdown()

取得件数: 458
取得件数: 461
取得件数: 457
取得件数: 460
取得件数: 457
取得件数: 458
取得件数: 458
取得件数: 459
取得件数: 458
取得件数: 458


'|    |   close |    market_cap |   shares_outstanding |   higher_rate_par_52_weeks |   lower_rate_par_52_weeks |     beta |\n|---:|--------:|--------------:|---------------------:|---------------------------:|--------------------------:|---------:|\n|  0 |    2.76 | nan           |        nan           |                       7.92 |                      1.8  | 0.720323 |\n|  1 |    3.68 |   1.27115e+08 |          3.4542e+07  |                       4.12 |                      1    | 0.45397  |\n|  2 |    2.96 |   1.31892e+08 |          4.45581e+07 |                       3.08 |                      0.52 | 0.512942 |\n|  3 |    1.28 |   1.83686e+08 |          1.43505e+08 |                       1.56 |                      0.48 | 0.339089 |\n|  4 |    0.64 |   1.54763e+08 |          2.41818e+08 |                       1.88 |                      0.56 | 0.474929 |'

In [11]:
"""
◆ 2. 財務データ（Financials）+ EPS（Earnings Per Share）+ PBR（Price-to-Book Ratio）
• 売上高（Revenue）
• 営業利益（Operating Income）
• 純利益（Net Income）
• EBITDA（企業による）
• 総資産（Total Assets）
• 総負債（Total Liabilities）
• 現金（Cash）
• 希釈EPS（Diluted EPS）
• 基本EPS（Basic EPS）
• 営業キャッシュフロー（Operating Cash Flow）
• フリーキャッシュフロー（Free Cash Flow）
"""
financial_df = financial.calc_financial(
    code = code,
    market = market,
)
financial_df.to_markdown()

取得件数: 2042


'|    | date                |   revenue |      earnings |   total_assets |    total_debt |   cash_and_cash_equivalents |       EBITDA |   operating_income |   basic_eps |   diluted_eps |   operating_cash_flow |   free_cash_flow |\n|---:|:--------------------|----------:|--------------:|---------------:|--------------:|----------------------------:|-------------:|-------------------:|------------:|--------------:|----------------------:|-----------------:|\n|  0 | 2021-12-31 00:00:00 |       nan | nan           |  nan           | nan           |               nan           |  0           |      nan           |      nan    |        nan    |         nan           |    nan           |\n|  1 | 2022-12-31 00:00:00 |         0 |  -2.02935e+07 |    1.23737e+08 |   1.841e+07   |                 1.60133e+07 | -2.01934e+07 |       -1.99315e+07 |       -0.6  |         -0.6  |          -1.77853e+07 |     -1.78067e+07 |\n|  2 | 2023-12-31 00:00:00 |         0 |  -1.33862e+07 |    1.17683e+08 |   1.6

In [12]:
# PBR（Price-to-Book Ratio）やROE（Return on Equity）などの投資指標を計算
financial.calc_stock_investment_indicators(code=code, market=market).to_markdown()

取得件数: 457


'|    | date                |   EV | reason   |        BPS |        PBR |        ROE |   operating_income |   basic_eps |   diluted_eps |\n|---:|:--------------------|-----:|:---------|-----------:|-----------:|-----------:|-------------------:|------------:|--------------:|\n|  0 | 2021-12-31 00:00:00 |  nan | no_price | nan        | nan        | nan        |      nan           |      nan    |        nan    |\n|  1 | 2022-12-31 00:00:00 |  nan | no_price | nan        | nan        |  -0.19497  |       -1.99315e+07 |       -0.6  |         -0.6  |\n|  2 | 2023-12-31 00:00:00 |  nan | no_price | nan        | nan        |  -0.134588 |       -1.24868e+07 |       -0.37 |         -0.37 |\n|  3 | 2024-12-31 00:00:00 |  nan | nan      |   1.10985  |   0.612696 |  -0.123024 |       -1.71698e+07 |       -0.3  |         -0.3  |\n|  4 | 2025-12-31 00:00:00 |  nan | nan      |   0.995287 |   1.75829  |  -0.170319 |       -3.86662e+07 |       -0.24 |         -0.24 |'

In [13]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://nexgold.com/wp-content/uploads/2026/03/NEXG-Consolidated-Financial-Statements_December-31-2025.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/NEXG-Consolidated-Financial-Statements_December-31-2025.pdf.md


'/workspace/data/NEXG-Consolidated-Financial-Statements_December-31-2025.pdf.md'

In [14]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://nexgold.com/wp-content/uploads/2026/03/NEXG-MDA-December-31-2025.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/NEXG-MDA-December-31-2025.pdf.md


'/workspace/data/NEXG-MDA-December-31-2025.pdf.md'

In [ ]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="",
    directory_path="/workspace/data",
)
md_file_path

In [1]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://spartanmetals.com/wp-content/uploads/2025/12/Spartan-Metals-FS-Q3-2025-Sept-30-25-Final-11-28-25-Final-Sedar.pdf",
    directory_path="/workspace/data",
)
md_file_path

NameError: name 'pdf_to_md' is not defined

In [ ]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="",
    directory_path="/workspace/data",
)
md_file_path